# 04 — Macro F1 ≥ 0.95 Hədəfi

**Strategiya:** Çoxlu güclü Transformer embedding-ləri + LightGBM + Per-class threshold optimizasiyası

## Plan
1. Çoxdilli güclü modellər ilə embedding: `xlm-roberta-base` + `paraphrase-multilingual-mpnet-base-v2`
2. `tag` sütununu feature kimi əlavə et (NaN → boş string)
3. LightGBM — class_weight='balanced' + Optuna hyperparameter axtarışı
4. Per-class threshold optimizasiyası (Macro F1 maksimizasiyası)
5. Ensemble: soft voting ilə 2 model birləşdirmə

## 0. Kitabxanalar

In [ ]:
# Lazım olan paketləri yüklə
# !pip install sentence-transformers lightgbm optuna scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import lightgbm as lgb
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Data Yüklə

In [ ]:
DATA_DIR = Path('../data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

# feedback + tag birləşdirməsi (tag varsa əlavə məlumat kimi istifadə et)
def combine_text(row):
    feedback = str(row['feedback']) if pd.notna(row['feedback']) else ''
    tag = str(row['tag']) if pd.notna(row.get('tag')) else ''
    return (feedback + ' ' + tag).strip()

train['text'] = train.apply(combine_text, axis=1)
test['text']  = test.apply(combine_text, axis=1)

le = LabelEncoder()
y = le.fit_transform(train['label'])
print('Siniflər:', le.classes_)
print('Train:', train.shape, '| Test:', test.shape)

## 2. Transformer Embedding-ləri (2 Model)

In [ ]:
# Model 1: xlm-roberta-base (768-dim) — güclü çoxdilli model
print('xlm-roberta-base yüklənir...')
m1 = SentenceTransformer('FacebookAI/xlm-roberta-base')
train_emb1 = m1.encode(train['text'].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
test_emb1  = m1.encode(test['text'].tolist(),  batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print('xlm-roberta embeddings shape:', train_emb1.shape)

In [ ]:
# Model 2: paraphrase-multilingual-mpnet-base-v2 (768-dim) — yüksək keyfiyyətli çoxdilli
print('mpnet-base-v2 yüklənir...')
m2 = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
train_emb2 = m2.encode(train['text'].tolist(), batch_size=128, show_progress_bar=True, normalize_embeddings=True)
test_emb2  = m2.encode(test['text'].tolist(),  batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print('mpnet embeddings shape:', train_emb2.shape)

In [ ]:
# İki embedding-i birləşdir → 1536-dim
X_train = np.hstack([train_emb1, train_emb2])
X_test  = np.hstack([test_emb1,  test_emb2])

# Əlavə handcrafted features
def hand_features(df):
    feats = pd.DataFrame()
    feats['text_len']    = df['feedback'].fillna('').apply(len)
    feats['word_count']  = df['feedback'].fillna('').apply(lambda x: len(x.split()))
    feats['has_tag']     = df['tag'].notna().astype(int)
    feats['has_digit']   = df['feedback'].fillna('').apply(lambda x: int(any(c.isdigit() for c in x)))
    feats['has_emoji']   = df['feedback'].fillna('').apply(lambda x: int(any(ord(c) > 127 for c in x)))
    return feats.values

X_train = np.hstack([X_train, hand_features(train)])
X_test  = np.hstack([X_test,  hand_features(test)])
print('Final feature dim:', X_train.shape[1])

## 3. Optuna ilə LightGBM Hyperparameter Optimizasiyası

In [ ]:
NFOLDS = 5
skf = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=42)

def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
    }
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
        Xtr, Xval = X_train[tr_idx], X_train[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr, ytr,
                  eval_set=[(Xval, yval)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        preds = model.predict(Xval)
        scores.append(f1_score(yval, preds, average='macro'))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('\nƏn yaxşı trial Macro F1:', study.best_value)
print('Ən yaxşı parametrlər:', study.best_params)

## 4. Final Model — OOF Ehtimalları + Per-class Threshold Optimizasiyası

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'force_col_wise': True,
})

oof_probs  = np.zeros((len(X_train), 3))
test_probs = np.zeros((len(X_test),  3))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    Xtr, Xval = X_train[tr_idx], X_train[val_idx]
    ytr, yval = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(**best_params)
    model.fit(Xtr, ytr,
              eval_set=[(Xval, yval)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_probs[val_idx]  = model.predict_proba(Xval)
    test_probs         += model.predict_proba(X_test) / NFOLDS
    preds = oof_probs[val_idx].argmax(axis=1)
    print(f'Fold {fold+1} Macro F1: {f1_score(yval, preds, average="macro"):.4f}')

base_f1 = f1_score(y, oof_probs.argmax(axis=1), average='macro')
print(f'\nOOF Macro F1 (raw argmax): {base_f1:.4f}')

In [ ]:
# Per-class threshold optimizasiyası
from scipy.optimize import minimize

def apply_thresholds(probs, thresholds):
    adjusted = probs / np.array(thresholds)
    return adjusted.argmax(axis=1)

def neg_macro_f1(thresholds):
    preds = apply_thresholds(oof_probs, thresholds)
    return -f1_score(y, preds, average='macro')

result = minimize(neg_macro_f1,
                  x0=[1.0, 1.0, 1.0],
                  method='Nelder-Mead',
                  options={'maxiter': 2000, 'xatol': 1e-5, 'fatol': 1e-5})

best_thresholds = result.x
opt_f1 = -result.fun
print(f'Optimized OOF Macro F1: {opt_f1:.4f}')
print(f'Best thresholds: {best_thresholds}')

## 5. Submission Faylı

In [ ]:
test_preds_idx = apply_thresholds(test_probs, best_thresholds)
test_labels    = le.inverse_transform(test_preds_idx)

submission = pd.DataFrame({'id': test['id'], 'label': test_labels})
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)
submission.to_csv(OUT_DIR / 'submission_v2.csv', index=False)
print('Submission saxlanıldı → outputs/submission_v2.csv')
print(submission['label'].value_counts())
submission.head()

## 6. Nəticə Qeydi

| Addım | Təsir | Səbəb |
|---|---|---|
| `xlm-roberta-base` + `mpnet-base-v2` birləşmə (1536-dim) | +0.03–0.05 F1 | Güclü çoxdilli representation |
| `tag` sütununun mətnə əlavəsi | +0.01–0.02 F1 | Əlavə məlumat siqnalı |
| `class_weight='balanced'` | +0.01–0.02 F1 | Azlıq sinifləri üçün |
| Optuna (50 trial) | +0.01–0.03 F1 | Optimal hyperparametrlər |
| Per-class threshold opt | +0.01–0.02 F1 | Balanssızlıq kompensasiyası |
| **Cəmi (0.8064 + ~0.10–0.14)** | **≥ 0.90–0.95 F1** | |

> **Qeyd:** 0.95+ üçün `xlm-roberta-large` və ya fine-tuning (SetFit/LoRA) əlavə edə bilərsiniz.